#Integrantes:
  * Nicolás Fonseca
  * Bastián Rubio

# 1.- Importación de librerías y carga del dataset
Comenzamos importando la librería necesaria y cargando los datos de la Play Store.

In [76]:
import pandas as pd

# Cargamos el dataset
df = pd.read_csv('/content/googleplaystore.csv')

# 2.- Exploración Inicial
Visualizamos la estructura de las columnas, la cantidad y el tipo de datos para entender los problemas a resolver.

In [77]:
# Vista previa de las primeras 5 filas
display(df.head())

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [78]:
# Información general del dataframe
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  object 
 1   Category        10841 non-null  object 
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  object 
 4   Size            10841 non-null  object 
 5   Installs        10841 non-null  object 
 6   Type            10840 non-null  object 
 7   Price           10841 non-null  object 
 8   Content Rating  10840 non-null  object 
 9   Genres          10841 non-null  object 
 10  Last Updated    10841 non-null  object 
 11  Current Ver     10833 non-null  object 
 12  Android Ver     10838 non-null  object 
dtypes: float64(1), object(12)
memory usage: 1.1+ MB


In [79]:
# Verificación de calidad: nulos, duplicados y tamaño original
print(f'Columnas con datos nulos: \n{df.isnull().sum()}\n')
print(f'Filas duplicadas en todo el dataset: {df.duplicated().sum()}\n')
print(f'Total de filas dataset sin editar: {len(df)}\n')

print("--- Duplicados por columna ---")
for col in df.columns:
    n_duplicados = df[col].duplicated().sum()
    print(f"{col:20} → {n_duplicados} duplicados")

Columnas con datos nulos: 
App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64

Filas duplicadas en todo el dataset: 483

Total de filas dataset sin editar: 10841

--- Duplicados por columna ---
App                  → 1181 duplicados
Category             → 10807 duplicados
Rating               → 10800 duplicados
Reviews              → 4839 duplicados
Size                 → 10379 duplicados
Installs             → 10819 duplicados
Type                 → 10837 duplicados
Price                → 10748 duplicados
Content Rating       → 10834 duplicados
Genres               → 10721 duplicados
Last Updated         → 9463 duplicados
Current Ver          → 8008 duplicados
Android Ver          → 10807 duplicados


> **Observación:** La columna `App` tiene muchos duplicados. Comprobamos que estos duplicados se repiten, por ende es posible eliminarlos sin perder información sensible. Es importante hacerlo ahora para que no afecten los cálculos estadísticos posteriores.

# 3.- Eliminación de duplicados
La columna `App` tiene muchos duplicados. Los eliminamos ahora para que no sesguen los cálculos estadísticos posteriores (como la mediana).

In [80]:
# Eliminamos duplicados basados en el nombre de la aplicación
df.drop_duplicates(subset=["App"], inplace=True)

# Comprobamos que ya no existen duplicados en 'App'
print(f"Duplicados restantes en 'App': {df['App'].duplicated().sum()}")

Duplicados restantes en 'App': 0


# 4.- Imputación de datos faltantes en Rating
Rellenamos los datos nulos en la columna `Rating` utilizando la mediana del dataset limpio de duplicados.

In [81]:
# Imputamos los valores nulos de Rating con la mediana
df['Rating'] = df['Rating'].fillna(df['Rating'].median())

# Verificamos que ya no hay nulos y vemos los estadísticos básicos
print("Nulos en Rating:", df['Rating'].isnull().sum(), "\n")
print(df['Rating'].describe())

Nulos en Rating: 0 

count    9660.000000
mean        4.193975
std         0.518732
min         1.000000
25%         4.000000
50%         4.300000
75%         4.500000
max        19.000000
Name: Rating, dtype: float64


# 5.- Limpieza de columnas Installs y Price
Eliminamos los símbolos (como +, , y $) para poder convertir estas columnas a formato numérico y realizar cálculos.

In [82]:
# Limpieza de Installs
df['Installs'] = df['Installs'].str.replace('+', '', regex=False).str.replace(',', '', regex=False)
df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')

# Limpieza de Price (eliminamos el símbolo $)
df['Price'] = df['Price'].str.replace('$', '', regex=False)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

print("Tipos de datos actualizados:")
print(df[['Installs', 'Price']].dtypes)

Tipos de datos actualizados:
Installs    float64
Price       float64
dtype: object


# 6.- Transformación de Fechas (Last Updated)
Convertimos la columna `Last Updated` a un formato de tiempo real (datetime) para su correcto manejo.

In [83]:
# Convertimos el texto a datetime
df['Last Updated'] = pd.to_datetime(df['Last Updated'], errors='coerce')

display(df[['App', 'Last Updated']].head())

,App,Last Updated
0,Photo Editor & Candy Camera & Grid & ScrapBook,2018-01-07
1,Coloring book moana,2018-01-15
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",2018-08-01
3,Sketch - Draw & Paint,2018-06-08
4,Pixel Draw - Number Art Coloring Book,2018-06-20


# 7.- Tratamiento de Outliers en Rating
Utilizamos el Rango Intercuartílico (IQR) para filtrar valores atípicos extremos en las valoraciones.

In [84]:
# Calculamos los cuartiles y el rango intercuartílico (IQR) para la columna Rating
Q1 = df['Rating'].quantile(0.25)
Q3 = df['Rating'].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

# Sobrescribimos el df conservando solo los valores dentro de los límites normales
df = df[(df['Rating'] >= limite_inferior) & (df['Rating'] <= limite_superior)]

print(f"Filas resultantes tras eliminar outliers: {len(df)}")

Filas resultantes tras eliminar outliers: 9167


In [85]:
print(f'Columnas con datos nulos: \n{df.isnull().sum()}\n')

Columnas con datos nulos: 
App               0
Category          0
Rating            0
Reviews           0
Size              0
Installs          0
Type              1
Price             0
Content Rating    0
Genres            0
Last Updated      0
Current Ver       7
Android Ver       2
dtype: int64



In [ ]:
df[df['Last Updated'].isnull()][['App','Android Ver', 'Last Updated', 'Installs']]

,App,Android Ver,Last Updated,Installs
10472,Life Made WI-Fi Touchscreen Photo Frame,NaN,NaN,NaN


In [ ]:
df[df['Content Rating'].isnull()][['App','Android Ver', 'Last Updated', 'Installs']]

,App,Android Ver,Last Updated,Installs
10472,Life Made WI-Fi Touchscreen Photo Frame,NaN,NaN,NaN


In [ ]:
df[df['Price'].isnull()][['App','Android Ver', 'Last Updated', 'Installs']]

,App,Android Ver,Last Updated,Installs
10472,Life Made WI-Fi Touchscreen Photo Frame,NaN,NaN,NaN


In [ ]:
df[df['Installs'].isnull()][['App','Android Ver', 'Last Updated', 'Installs']]

,App,Android Ver,Last Updated,Installs
10472,Life Made WI-Fi Touchscreen Photo Frame,NaN,NaN,NaN


In [ ]:
df[df['Type'].isnull()][['App','Android Ver', 'Last Updated', 'Installs']]

,App,Android Ver,Last Updated,Installs
9148,Command & Conquer: Rivals,Varies with device,28/06/2018,0.0


In [ ]:
#Ver solo la columna Android Ver con su índice con los valores NaN y que App es
df[df['Android Ver'].isnull()][['App', 'Android Ver', 'Last Updated', 'Installs']]

,App,Android Ver,Last Updated,Installs
4453,[substratum] Vacuum: P,NaN,20/07/2018,1000.0
4490,Pi Dark [substratum],NaN,27/03/2018,10000.0
10472,Life Made WI-Fi Touchscreen Photo Frame,NaN,NaN,NaN


In [ ]:
#Ver solo la columna Android Ver con su índice con los valores NaN y que App es
df[df['Current Ver'].isnull()][['App', 'Current Ver', 'Last Updated']]

,App,Current Ver,Last Updated
15,Learn To Draw Kawaii Characters,NaN,06/06/2018
1553,Market Update Helper,NaN,12/02/2013
6322,Virtual DJ Sound Mixer,NaN,10/05/2017
6803,BT Master,NaN,06/11/2016
7333,Dots puzzle,NaN,18/04/2018
7407,Calculate My IQ,NaN,03/04/2017
7730,UFO-CQ,NaN,04/07/2016
10342,La Fe de Jesus,NaN,31/01/2017


In [ ]:
df[df['Type'].isnull()][['App', 'Type', 'Last Updated']]

,App,Type,Last Updated
9148,Command & Conquer: Rivals,NaN,28/06/2018


* Verificamos que los datos nulos restantes son aplicaciones desactualizadas y con descargar inferiores a 10.000. Estas pueden ser eliminadas sin afectar a la información importante.

In [86]:
#eliminamos los datos verificados
df.dropna(subset=['Type','Current Ver','Android Ver'], inplace=True)

# 8.- Verificación Final del Dataset
Comprobamos la estructura final después de todo el pipeline de limpieza para asegurar que los datos estén listos para el análisis.

In [87]:
# Validamos la info general y la estadística descriptiva final
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9157 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   App             9157 non-null   object        
 1   Category        9157 non-null   object        
 2   Rating          9157 non-null   float64       
 3   Reviews         9157 non-null   object        
 4   Size            9157 non-null   object        
 5   Installs        9157 non-null   float64       
 6   Type            9157 non-null   object        
 7   Price           9157 non-null   float64       
 8   Content Rating  9157 non-null   object        
 9   Genres          9157 non-null   object        
 10  Last Updated    9157 non-null   datetime64[ns]
 11  Current Ver     9157 non-null   object        
 12  Android Ver     9157 non-null   object        
dtypes: datetime64[ns](1), float64(3), object(9)
memory usage: 1001.5+ KB
